# CGR: Constraint-Aware Generative Re-ranking

**Paper:** *Constraint-Aware Generative Re-ranking for Multi-Objective Optimization in Advertising Feeds* (Bilibili, arXiv:2603.04227)

This notebook walks through the key components of the current CGR-style implementation in this repository:

1. **Setup** -- imports, dimensions, and device configuration
2. **Domain types** -- items, constraints, candidate sets
3. **Model architecture** -- dual hierarchical attention + PLE fusion + exposure/click heads
4. **Computing list-level reward** -- $R(A) = \sum_i (V_i + N_i - P_i)$
5. **Training** -- weighted multi-task BCE in logits space
6. **Two-stage inference** -- Stage I insertion + Stage II bounded decoding
7. **Full pipeline** -- one-call inference with the implemented search procedure
8. **Pruning comparison** -- with vs. without reward upper-bound pruning
9. **Constraint experiments** -- changing business rules
10. **Beam search** -- alternative inference for larger $K$
11. **Two-stage vs. beam search comparison**

The emphasis here is on understanding how the repository works today. Where the paper makes stronger optimality claims than this implementation supports, the notebook calls that out explicitly.

In [18]:
import torch
from torch.utils.data import DataLoader

from cgr.data_types import AdConstraints, CandidateSet, Item, ItemType
from cgr.inference import cgr_inference, stage1_constrained_insertion, stage2_bounded_decoding
from cgr.model import CGRModel, ExposureClickHead
from cgr.train import CGRTrainer, TrainConfig

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ITEM_EMB_DIM = 64
USER_EMB_DIM = 32
CTX_EMB_DIM = 16

print(f"Device: {device}")
print(
    f"Embedding dims: item={ITEM_EMB_DIM}, user={USER_EMB_DIM}, context={CTX_EMB_DIM}"
)

Device: cpu
Embedding dims: item=64, user=32, context=16


## 1. Domain Types — Items, Constraints, Candidate Sets

CGR operates on **Items** that carry pre-computed embeddings from upstream models (DLRM item tower, user tower, context encoder) plus scalar signals from the ad auction and business policy.

Key classes:
- `Item` — a single feed candidate (organic or ad) with upstream embeddings + auction scalars
- `AdConstraints` — business rules: max ads per page (K), minimum spacing (Δ), position bounds, etc.
- `CandidateSet` — the organic list + ad candidates for a single user request

In [19]:
organic_item = Item(
    item_id=1,
    item_type=ItemType.ORGANIC,
    item_emb=torch.randn(ITEM_EMB_DIM),
    user_emb=torch.randn(USER_EMB_DIM),
    context_emb=torch.randn(CTX_EMB_DIM),
    engagement_score=1.2,
)

ad_item = Item(
    item_id=100,
    item_type=ItemType.LARGE_AD,
    item_emb=torch.randn(ITEM_EMB_DIM),
    user_emb=torch.randn(USER_EMB_DIM),
    context_emb=torch.randn(CTX_EMB_DIM),
    bid=4.0,
    engagement_score=0.7,
    ad_penalty_weight=0.4,
)

preview_candidates = CandidateSet(
    organic_items=[organic_item],
    ad_candidates=[ad_item],
)

print(f"Organic item is_ad: {organic_item.is_ad}")
print(f"Ad item is_ad: {ad_item.is_ad}")
print(f"Ad item visual slots: {ad_item.visual_slots}")
print(
    f"CandidateSet: {preview_candidates.num_organic} organic + {preview_candidates.num_ads} ad"
)

Organic item is_ad: False
Ad item is_ad: True
Ad item visual slots: 2
CandidateSet: 1 organic + 1 ad


In [20]:
# --- Define business constraints (Section 3.4) ---
constraints = AdConstraints(
    max_ads_per_list=2,    # K: at most 2 ads per page
    min_ad_spacing=3,      # Δ: ads must be ≥3 positions apart
    min_ad_position=1,     # ads cannot appear at position 0 (top of feed)
    max_ad_position=8,     # ads cannot appear past position 8
    max_large_ads=1,       # at most 1 large-format ad
    ad_density_limit=0.3,  # at most 30% of the page can be ads
)

print("Feasible ad positions for a 10-item list:", constraints.get_feasible_ad_positions(10))
print()

# Check spacing: positions [2, 5] with min_spacing=3 → OK (5-2=3 ≥ 3)
print("Spacing check [2, 5]:", constraints.check_spacing([2, 5]))
# Check spacing: positions [2, 4] with min_spacing=3 → FAIL (4-2=2 < 3)
print("Spacing check [2, 4]:", constraints.check_spacing([2, 4]))
# Load check: 2 ads with K=2 → OK
print("Load check (2 ads):", constraints.check_load(2))
# Load check: 3 ads with K=2 → FAIL
print("Load check (3 ads):", constraints.check_load(3))

Feasible ad positions for a 10-item list: [1, 2, 3, 4, 5, 6, 7, 8]

Spacing check [2, 5]: True
Spacing check [2, 4]: False
Load check (2 ads): True
Load check (3 ads): False


## 2. Model Architecture

The repository model follows the paper's high-level idea but is best read as a paper-inspired implementation rather than a line-by-line reproduction.

Current structure:
- Adapter layers project pre-computed item, user, and context embeddings into a shared hidden space.
- Two stacked attention streams are maintained: one oriented toward exposure and one toward click.
- A PLE-style fusion block combines shared and task-specific experts into two task heads.
- `ExposureClickHead` predicts per-position exposure and click values for the specific ordered list you pass in.

Those predictions are not the final business reward by themselves. The list reward is computed afterward by combining `p_exp` and `p_clk` with bids, engagement scores, and ad-penalty weights.

In [21]:
from cgr.model import CGRModel

model = CGRModel(
    item_emb_dim=ITEM_EMB_DIM,    # must match upstream item tower output dim
    user_emb_dim=USER_EMB_DIM,    # must match upstream user tower output dim
    context_emb_dim=CTX_EMB_DIM,  # must match upstream context encoder output dim
    d_model=32,                    # internal hidden dimension
    n_heads=4,                     # attention heads per MHA layer
    n_layers=2,                    # number of HierarchicalAttentionBlock layers
    max_list_len=11,               # maximum sequence length L (paper uses 11)
    dropout=0.1,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel structure:\n{model}")

Total parameters:     121,398
Trainable parameters: 121,398

Model structure:
CGRModel(
  (item_adapter): Linear(in_features=64, out_features=32, bias=True)
  (user_adapter): Linear(in_features=32, out_features=32, bias=True)
  (context_adapter): Linear(in_features=16, out_features=32, bias=True)
  (position_emb): Embedding(11, 32)
  (item_type_emb): Embedding(4, 32)
  (input_fusion): Sequential(
    (0): Linear(in_features=128, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (exp_attn_blocks): ModuleList(
    (0-1): 2 x HierarchicalAttentionBlock(
      (shared_self_attn): MultiHeadSelfAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
      )
      (exp_self_attn): MultiHeadSelfAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
      )
   

In [22]:
B, L = 2, 5
item_embs = torch.randn(B, L, ITEM_EMB_DIM, device=device)
user_embs = torch.randn(B, L, USER_EMB_DIM, device=device)
context_embs = torch.randn(B, L, CTX_EMB_DIM, device=device)
item_types = torch.tensor(
    [
        [ItemType.ORGANIC.value, ItemType.AD.value, ItemType.ORGANIC.value, ItemType.ECOM_AD.value, ItemType.ORGANIC.value],
        [ItemType.ORGANIC.value, ItemType.ORGANIC.value, ItemType.LARGE_AD.value, ItemType.ORGANIC.value, ItemType.AD.value],
    ],
    device=device,
)

model = model.to(device)
p_exp, p_clk = model(item_embs, user_embs, context_embs, item_types)

print(f"p_exp shape: {tuple(p_exp.shape)}")
print(f"p_clk shape: {tuple(p_clk.shape)}")
print("Example exposure probabilities:", p_exp[0].detach().cpu())
print("Example click probabilities:", p_clk[0].detach().cpu())

p_exp shape: (2, 5)
p_clk shape: (2, 5)
Example exposure probabilities: tensor([0.5482, 0.4800, 0.5310, 0.5072, 0.5621])
Example click probabilities: tensor([0.5052, 0.4914, 0.4639, 0.5180, 0.5077])


## 3. Computing List-Level Reward

The reward R(A) combines three per-position components (Eqs. 17-20):

| Component | Formula | Meaning |
|-----------|---------|---------|
| V_i (monetization) | `p_clk * p_exp * bid` (CPA) or `p_exp * bid` (CPM) | Ad revenue |
| N_i (engagement) | `p_exp * p_clk * engagement_score` | User satisfaction |
| P_i (penalty) | `ad_penalty_weight * p_exp` | Cost of showing an ad |

**R(A) = Σ(V_i + N_i − P_i)** — the model maximises revenue + engagement while penalising ad overexposure.

In [23]:
bids = torch.tensor([[0.0, 2.5, 0.0, 1.8, 0.0], [0.0, 0.0, 4.0, 0.0, 2.2]], device=device)
engagement_scores = torch.tensor(
    [[1.1, 0.5, 1.3, 0.6, 1.2], [1.0, 1.1, 0.8, 1.4, 0.4]], device=device
)
ad_penalty_weights = torch.tensor(
    [[0.0, 0.3, 0.0, 0.2, 0.0], [0.0, 0.0, 0.5, 0.0, 0.3]], device=device
)
is_cpa = (item_types != ItemType.ORGANIC.value).float()

reward = ExposureClickHead.compute_reward(
    p_exp, p_clk, bids, engagement_scores, ad_penalty_weights, is_cpa
)
print("List-level reward per sample:", reward.detach().cpu())

List-level reward per sample: tensor([2.0601, 2.4040])


## 4. Training on Logged Impression Data

In production, training data consists of logged impression sequences:
- pre-computed embeddings from upstream retrieval/ranking towers
- binary exposure labels indicating whether an item was actually shown
- binary click labels indicating whether the shown item was clicked

In this repository, training is done in logits space with `BCEWithLogitsLoss`:

$L = \lambda_{exp} \cdot L_{exp} + \lambda_{clk} \cdot L_{clk}$

The click task also supports `click_positive_class_weight`, because clicks are usually much rarer than exposures.

In [24]:
n_samples = 128
list_len = 10

train_data = [
    {
        "item_embs": torch.randn(list_len, ITEM_EMB_DIM),
        "user_embs": torch.randn(list_len, USER_EMB_DIM),
        "context_embs": torch.randn(list_len, CTX_EMB_DIM),
        "item_types": torch.randint(0, 4, (list_len,)),
        "exp_labels": torch.randint(0, 2, (list_len,), dtype=torch.float32),
        "clk_labels": torch.randint(0, 2, (list_len,), dtype=torch.float32),
    }
    for _ in range(n_samples)
]

loader = DataLoader(train_data, batch_size=32, shuffle=True)
print(f"Synthetic training pages: {len(train_data)}")
print(f"Mini-batches per epoch: {len(loader)}")

Synthetic training pages: 128
Mini-batches per epoch: 4


In [25]:
# --- Train the model ---
config = TrainConfig(
    lr=1e-3,
    weight_decay=1e-5,
    lambda_exp=1.0,
    lambda_clk=1.0,
    click_positive_class_weight=2.0,
    epochs=5,
    batch_size=32,
    grad_clip=1.0,
)

# Re-initialise model for a clean training run
model = CGRModel(
    item_emb_dim=ITEM_EMB_DIM,
    user_emb_dim=USER_EMB_DIM,
    context_emb_dim=CTX_EMB_DIM,
    d_model=32,
    n_heads=4,
    n_layers=2,
    max_list_len=11,
).to(device)
trainer = CGRTrainer(model, config, device)

print("Epoch  | Loss    | L_exp   | L_clk")
print("-------|---------|---------|--------")
for epoch in range(config.epochs):
    metrics = trainer.train_epoch(loader)
    print(f"  {epoch+1:2d}   | {metrics['loss']:.4f}  | {metrics['loss_exp']:.4f}  | {metrics['loss_clk']:.4f}")

Epoch  | Loss    | L_exp   | L_clk
-------|---------|---------|--------
   1   | 1.7009  | 0.7010  | 0.9999
   2   | 1.6576  | 0.6927  | 0.9649
   3   | 1.6449  | 0.6899  | 0.9551
   4   | 1.6361  | 0.6862  | 0.9499
   5   | 1.6300  | 0.6839  | 0.9461


## 5. Two-Stage Inference

The repository inference flow has two parts:

**Stage I -- constrained insertion**
- Try feasible single-ad insertions into the organic list.
- Score them with the current model.
- Keep the best single-ad result, or the no-ad organic baseline if that scores higher.

**Stage II -- bounded decoding**
- Starting from the original organic list, enumerate the list families implemented here: no-ad, single-ad, double-ad, and large-ad variants.
- Filter candidates against hard structural constraints before scoring.
- Optionally use an upper-bound heuristic to skip candidates that are unlikely to beat the current best.

This is a practical bounded-search implementation for small $K$. It should be read as "best sequence found by this search" rather than as a general proof-backed optimal solver.

In [26]:
from cgr.inference import cgr_inference, stage1_constrained_insertion, stage2_bounded_decoding

# --- Build a realistic candidate set ---
def make_item(item_id, item_type, bid=0.0, engagement=1.0, penalty=0.0):
    return Item(
        item_id=item_id,
        item_type=item_type,
        item_emb=torch.randn(ITEM_EMB_DIM),
        user_emb=torch.randn(USER_EMB_DIM),
        context_emb=torch.randn(CTX_EMB_DIM),
        bid=bid,
        engagement_score=engagement,
        ad_penalty_weight=penalty,
    )

# 8 organic items (pre-ranked by upstream DLRM)
organic_items = [make_item(i, ItemType.ORGANIC, engagement=1.0 + 0.1 * i) for i in range(8)]

# 4 ad candidates from the auction system
ad_candidates = [
    make_item(100, ItemType.AD,       bid=2.5, engagement=0.5, penalty=0.3),
    make_item(101, ItemType.AD,       bid=3.0, engagement=0.3, penalty=0.4),
    make_item(102, ItemType.LARGE_AD, bid=5.0, engagement=0.8, penalty=0.5),
    make_item(103, ItemType.ECOM_AD,  bid=1.5, engagement=0.6, penalty=0.2),
]

candidates = CandidateSet(organic_items=organic_items, ad_candidates=ad_candidates)
print(f"Candidate set: {candidates.num_organic} organic + {candidates.num_ads} ads")

Candidate set: 8 organic + 4 ads


In [27]:
# --- Stage I: constrained single-ad insertion ---
print("=" * 60)
print("Stage I: Constrained Insertion")
print("=" * 60)

intermediate = stage1_constrained_insertion(
    model, organic_items, ad_candidates, constraints, device
)

print(f"Stage I reward: {intermediate.reward:.4f}")
print(f"Stage I ad positions: {intermediate.ad_positions}")
print(f"Stage I length: {len(intermediate.items)}")
for i, item in enumerate(intermediate.items):
    marker = f" <- {item.item_type.name} (bid={item.bid})" if item.is_ad else ""
    print(f"  [{i}] item_{item.item_id}{marker}")

Stage I: Constrained Insertion
Stage I reward: 4.6297
Stage I ad positions: [8]
Stage I length: 9
  [0] item_0
  [1] item_1
  [2] item_2
  [3] item_3
  [4] item_4
  [5] item_5
  [6] item_6
  [7] item_7
  [8] item_100 <- AD (bid=2.5)


In [28]:
# --- Stage II: bounded generative decoding ---
print("=" * 60)
print("Stage II: Bounded Generative Decoding")
print("=" * 60)

final = stage2_bounded_decoding(
    model, intermediate, ad_candidates, constraints, device, use_pruning=True
)

print(f"Stage II reward: {final.reward:.4f}")
print(f"Ad positions: {final.ad_positions}")
print(f"Sequence ({len(final.items)} items):")
for i, item in enumerate(final.items):
    marker = f" <- {item.item_type.name} (bid={item.bid})" if item.is_ad else ""
    print(f"  [{i}] item_{item.item_id}{marker}")

print(f"\nConstraints satisfied: {constraints.is_feasible(final.items)}")

Stage II: Bounded Generative Decoding
Stage II reward: 5.5262
Ad positions: [7]
Sequence (9 items):
  [0] item_0
  [1] item_1
  [2] item_2
  [3] item_3
  [4] item_4
  [5] item_5
  [6] item_6
  [7] item_102 <- LARGE_AD (bid=5.0)
  [8] item_7

Constraints satisfied: True


In [29]:
# --- Full inference in one call ---
result = cgr_inference(
    model=model,
    candidates=candidates,
    constraints=constraints,
    device=device,
    use_pruning=True,
)

print(f"Best reward found: {result.reward:.4f}")
print(f"Number of ads:     {result.num_ads}")
print(f"Ad positions:      {result.ad_positions}")
print(f"Constraints OK:    {constraints.is_feasible(result.items)}")
print("\nFinal feed:")
for i, item in enumerate(result.items):
    if item.is_ad:
        print(f"  [{i}] >> {item.item_type.name} (id={item.item_id}, bid={item.bid}, penalty={item.ad_penalty_weight})")
    else:
        print(f"  [{i}]    ORGANIC (id={item.item_id}, engagement={item.engagement_score})")

Best reward found: 5.6498
Number of ads:     2
Ad positions:      [1, 8]
Constraints OK:    True

Final feed:
  [0]    ORGANIC (id=0, engagement=1.0)
  [1] >> AD (id=100, bid=2.5, penalty=0.3)
  [2]    ORGANIC (id=1, engagement=1.1)
  [3]    ORGANIC (id=2, engagement=1.2)
  [4]    ORGANIC (id=3, engagement=1.3)
  [5]    ORGANIC (id=4, engagement=1.4)
  [6]    ORGANIC (id=5, engagement=1.5)
  [7]    ORGANIC (id=6, engagement=1.6)
  [8] >> AD (id=101, bid=3.0, penalty=0.4)
  [9]    ORGANIC (id=7, engagement=1.7000000000000002)


## 7. Comparing Pruning vs. No Pruning

Reward pruning uses an optimistic upper-bound estimate to skip candidates that look unable to beat the current best.

Two important practical notes for this repository:
- When the candidate space is already small, pruning can be slower because computing the bound itself costs extra model work.
- Matching rewards in the comparison below is good evidence that pruning did not change the selected result in this example, but the notebook should not describe that as a theorem-backed guarantee for the implementation.

In [30]:
import time

# With pruning
t0 = time.perf_counter()
result_pruned = cgr_inference(model, candidates, constraints, device, use_pruning=True)
t_pruned = time.perf_counter() - t0

# Without pruning
t0 = time.perf_counter()
result_no_prune = cgr_inference(model, candidates, constraints, device, use_pruning=False)
t_no_prune = time.perf_counter() - t0

print(f"With pruning:    reward={result_pruned.reward:.4f}  time={t_pruned*1000:.1f}ms")
print(f"Without pruning: reward={result_no_prune.reward:.4f}  time={t_no_prune*1000:.1f}ms")
print(f"\nRewards match: {abs(result_pruned.reward - result_no_prune.reward) < 1e-4}")
if t_pruned > t_no_prune:
    print("Pruning is slower here because the search space is small and the upper-bound check adds overhead.")
else:
    print("Pruning helps here because it removes enough candidates to offset the upper-bound overhead.")

With pruning:    reward=5.6498  time=433.7ms
Without pruning: reward=5.6498  time=59.7ms

Rewards match: True
Pruning is slower here because the search space is small and the upper-bound check adds overhead.


## 8. Experimenting with Different Constraints

This section shows how the chosen list changes when the structural business rules change.

The point is not that there is one universally best ad layout. The point is that the same model can be searched under different policy envelopes, producing different feasible winners.

In [31]:
# --- Strict: only 1 ad allowed ---
strict = AdConstraints(max_ads_per_list=1, min_ad_spacing=3, min_ad_position=1, max_ad_position=8)
result_strict = cgr_inference(model, candidates, strict, device)
print(f"Strict (K=1):  {result_strict.num_ads} ad(s), reward={result_strict.reward:.4f}, positions={result_strict.ad_positions}")

# --- Relaxed: up to 2 ads, smaller spacing ---
relaxed = AdConstraints(max_ads_per_list=2, min_ad_spacing=2, min_ad_position=1, max_ad_position=8)
result_relaxed = cgr_inference(model, candidates, relaxed, device)
print(f"Relaxed (Δ=2): {result_relaxed.num_ads} ad(s), reward={result_relaxed.reward:.4f}, positions={result_relaxed.ad_positions}")

# --- No ads allowed ---
no_ads = AdConstraints(max_ads_per_list=0)
result_no_ads = cgr_inference(model, candidates, no_ads, device)
print(f"No ads (K=0):  {result_no_ads.num_ads} ad(s), reward={result_no_ads.reward:.4f}")

print(f"\nAll constraints satisfied:")
print(f"  Strict:  {strict.is_feasible(result_strict.items)}")
print(f"  Relaxed: {relaxed.is_feasible(result_relaxed.items)}")
print(f"  No ads:  {no_ads.is_feasible(result_no_ads.items)}")

Strict (K=1):  1 ad(s), reward=5.4521, positions=[5]
Relaxed (Δ=2): 2 ad(s), reward=5.6498, positions=[1, 8]
No ads (K=0):  0 ad(s), reward=3.6312

All constraints satisfied:
  Strict:  True
  Relaxed: True
  No ads:  True


## 9. Beam Search -- Alternative Inference for Larger K

The two-stage pipeline is a bounded-search baseline that works well when K is small, but it does not scale gracefully to larger K because the number of feasible combinations grows quickly.

Beam search is an alternative that works for any K:

1. Start with the organic list (no ads)
2. At each step, try inserting each unused ad at each feasible position
3. Keep the top-B candidates (beam width)
4. Repeat up to K steps

Why beam search is interesting here: it trades exact coverage of a small search space for a controllable approximate search over a much larger space.

Trade-off: beam search can miss the global best if a good partial sequence is pruned early, but it handles K > 2 where exhaustive or near-exhaustive enumeration becomes expensive.

Large ads occupy 2 visual slots (`Item.visual_slots`), which beam search accounts for in density constraint checks.

### Beam Search Examples

Below we run beam search in two regimes:
- `K=2` to compare it with the two-stage bounded-search pipeline on the same problem size.
- `K=3` to show why a heuristic search can still be useful once the allowed number of ads grows beyond the small-$K$ setting the two-stage path was designed around.

Beam search keeps only the top-$B$ partial candidates at each expansion step, so quality depends on beam width and the scoring quality of partial states.

In [32]:
from cgr.inference import beam_search_inference

# --- Beam search with K=2 (same as two-stage for comparison) ---
result_beam = beam_search_inference(
    model, candidates, constraints, device, beam_width=10
)

print(f"Beam search (B=10, K=2):")
print(f"  Reward:      {result_beam.reward:.4f}")
print(f"  Num ads:     {result_beam.num_ads}")
print(f"  Ad positions: {result_beam.ad_positions}")
print(f"  Feasible:    {constraints.is_feasible(result_beam.items)}")
print(f"\nFeed:")
for i, item in enumerate(result_beam.items):
    if item.is_ad:
        print(f"  [{i}] >> {item.item_type.name} (id={item.item_id}, bid={item.bid}, slots={item.visual_slots})")
    else:
        print(f"  [{i}]    ORGANIC (id={item.item_id})")

Beam search (B=10, K=2):
  Reward:      6.5026
  Num ads:     2
  Ad positions: [5, 8]
  Feasible:    True

Feed:
  [0]    ORGANIC (id=0)
  [1]    ORGANIC (id=1)
  [2]    ORGANIC (id=2)
  [3]    ORGANIC (id=3)
  [4]    ORGANIC (id=4)
  [5] >> LARGE_AD (id=102, bid=5.0, slots=2)
  [6]    ORGANIC (id=5)
  [7]    ORGANIC (id=6)
  [8] >> AD (id=101, bid=3.0, slots=1)
  [9]    ORGANIC (id=7)


In [33]:
# --- Beam search with K=3 (two-stage can't do this) ---
constraints_k3 = AdConstraints(
    max_ads_per_list=3, min_ad_spacing=2, min_ad_position=1, max_ad_position=9
)

result_beam_k3 = beam_search_inference(
    model, candidates, constraints_k3, device, beam_width=10, max_ads=3
)

print(f"Beam search (B=10, K=3, spacing=2):")
print(f"  Reward:      {result_beam_k3.reward:.4f}")
print(f"  Num ads:     {result_beam_k3.num_ads}")
print(f"  Ad positions: {result_beam_k3.ad_positions}")
print(f"  Feasible:    {constraints_k3.is_feasible(result_beam_k3.items)}")
print(f"\nFeed:")
for i, item in enumerate(result_beam_k3.items):
    if item.is_ad:
        print(f"  [{i}] >> {item.item_type.name} (id={item.item_id}, bid={item.bid})")
    else:
        print(f"  [{i}]    ORGANIC (id={item.item_id})")

Beam search (B=10, K=3, spacing=2):
  Reward:      6.5026
  Num ads:     2
  Ad positions: [5, 8]
  Feasible:    True

Feed:
  [0]    ORGANIC (id=0)
  [1]    ORGANIC (id=1)
  [2]    ORGANIC (id=2)
  [3]    ORGANIC (id=3)
  [4]    ORGANIC (id=4)
  [5] >> LARGE_AD (id=102, bid=5.0)
  [6]    ORGANIC (id=5)
  [7]    ORGANIC (id=6)
  [8] >> AD (id=101, bid=3.0)
  [9]    ORGANIC (id=7)


## 10. Two-Stage vs. Beam Search Comparison

| | Two-stage (`cgr_inference`) | Beam search (`beam_search_inference`) |
|---|---|---|
| **Primary use case** | Small-$K$ bounded search in this repository | Larger-$K$ heuristic search |
| **Search style** | Enumerates the implemented list families for small $K$ | Expands partial candidates and keeps the top-$B$ |
| **Guarantee** | Best result found by the implemented bounded search | Can miss the best full sequence if a strong partial state is dropped |
| **Latency pattern** | A few batched evaluations over grouped candidates | Roughly beam width × search depth |
| **Large ads** | Handled as a separate candidate family | Handled through `visual_slots` in feasibility checks |

In [34]:
# --- Side-by-side timing comparison ---
import time

# Two-stage bounded search (K=2)
t0 = time.perf_counter()
r_twostage = cgr_inference(model, candidates, constraints, device, use_pruning=True)
t_twostage = time.perf_counter() - t0

# Beam search greedy (B=1)
t0 = time.perf_counter()
r_greedy = beam_search_inference(model, candidates, constraints, device, beam_width=1)
t_greedy = time.perf_counter() - t0

# Beam search (B=5)
t0 = time.perf_counter()
r_beam5 = beam_search_inference(model, candidates, constraints, device, beam_width=5)
t_beam5 = time.perf_counter() - t0

# Beam search (B=20)
t0 = time.perf_counter()
r_beam20 = beam_search_inference(model, candidates, constraints, device, beam_width=20)
t_beam20 = time.perf_counter() - t0

print("Method                    | Reward  | Ads | Time")
print("--------------------------|---------|-----|--------")
print(f"Two-stage bounded search  | {r_twostage.reward:.4f} |  {r_twostage.num_ads}  | {t_twostage*1000:.1f}ms")
print(f"Beam B=1 (greedy)         | {r_greedy.reward:.4f} |  {r_greedy.num_ads}  | {t_greedy*1000:.1f}ms")
print(f"Beam B=5                  | {r_beam5.reward:.4f} |  {r_beam5.num_ads}  | {t_beam5*1000:.1f}ms")
print(f"Beam B=20                 | {r_beam20.reward:.4f} |  {r_beam20.num_ads}  | {t_beam20*1000:.1f}ms")
print(
    f"\nAll feasible: two-stage={constraints.is_feasible(r_twostage.items)}, "
    f"B=1={constraints.is_feasible(r_greedy.items)}, "
    f"B=5={constraints.is_feasible(r_beam5.items)}, "
    f"B=20={constraints.is_feasible(r_beam20.items)}"
)

Method                    | Reward  | Ads | Time
--------------------------|---------|-----|--------
Two-stage bounded search  | 5.6498 |  2  | 498.7ms
Beam B=1 (greedy)         | 6.5026 |  2  | 33.8ms
Beam B=5                  | 6.5026 |  2  | 53.0ms
Beam B=20                 | 6.5026 |  2  | 62.8ms

All feasible: two-stage=True, B=1=True, B=5=True, B=20=True


## Summary

| Module | Paper Section | What it does in this repository |
|--------|:------------:|----------------------------------|
| `cgr.data_types` | 3 | `Item`, `AdConstraints`, `CandidateSet` -- typed feed objects and structural constraint checks |
| `cgr.model` | 4 | `CGRModel` -- dual attention streams, PLE fusion, and exposure/click prediction heads |
| `cgr.train` | 4 | `CGRTrainer` -- weighted multi-task training with `BCEWithLogitsLoss` |
| `cgr.inference` | 5, 6, 7 | Stage I insertion, Stage II bounded decoding, optional pruning, and beam search |

**Architecture**
- CGR consumes pre-computed upstream embeddings rather than raw sparse features.
- The ordered list is the input. `p_exp` and `p_clk` are per-position predictions for that specific arrangement.
- The model predicts exposure and click first; reward is computed afterward from bids, engagement, and penalty terms.

**Inference methods**
- `cgr_inference` is the repository's bounded-search path for small $K$.
- `beam_search_inference` is a heuristic alternative that can be used when the allowed number of ads grows.
- In practice, beam search can sometimes beat the two-stage path in this repo because the implementation is not a full paper-faithful optimal solver.

**Constraints enforced here**
- Ad load cap, minimum spacing, position bounds, large-ad cap, and density limit are all checked structurally during search.
- Non-structural constraints such as budget pacing, user-level frequency caps, or category exclusions are still upstream concerns.